# Track B, step 4: out-of-fold scores and the handoff files

Track A (the signals model) can use our text score as one of its inputs. But there's a trap: if our model scores rows it *trained on*, it gives them overconfident scores. Track A's model would then learn to over-trust the text score, and the trust wouldn't hold on 2026.

**The fix is out-of-fold scoring.** Split the training rows into 5 parts. Train on 4 and score the 5th, and rotate until every row has been scored by a model that never saw it. *Example:* a row in part 3 is scored by the model trained on parts 1, 2, 4 and 5.

**The four handoff files**, all in the team format `cve_id, text_score`:

| File | Model trained on | Scores | Used for |
|---|---|---|---|
| `text_oof_train.csv` | 5 folds of 2018-2024 (labels by Dec 31, 2024) | 2024 rows, out-of-fold | Track A training |
| `text_val_2025.csv` | all of 2018-2024 | 2025 validation set | Comparing approaches (copied from step 3) |
| `text_oof_final.csv` | 5 folds of 2018-2024 + 2025 (labels by Dec 31, 2025) | 2024-2025 rows, out-of-fold | Track A final training |
| `text_2026.csv` | all of 2018-2024 + 2025 (labels by Dec 31, 2025) | all 72,670 scoreboard rows | The final 2026 ranking |

The model settings are exactly step 3's (the code cells below are copied from it), so these scores behave like the one we validated.

**Runs best on an NVIDIA GPU:** 11 fine-tuning runs. Needs `data/trackb/texts.parquet` from step 3, and `text_val_2025_finetuned.csv` if step 3 has run.

## 1. Setup (same as step 3)

In [ ]:
import os, re, time, math, random, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

DATA = next(p for p in [Path("../data"), Path("data")] if p.exists()).resolve()
OUT = DATA / "trackb"
OUT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
SMOKE = os.environ.get("SMOKE") == "1"

CFG = dict(
    model="ibm-granite/granite-embedding-small-english-r2",
    max_length=256,        # covers 97% of descriptions in full (step 1)
    neg_per_pos=20,        # keep all positives + 20 random negatives each
    pos_weight=4.0,        # a missed positive costs 4x a false alarm in the loss
    lr=3e-5, batch_size=32, epochs=3, warmup=0.06, weight_decay=0.01,
    holdout=0.10,          # share of training rows held back to watch the loss
    seed=0,
)
if SMOKE:
    CFG.update(epochs=1, neg_per_pos=5)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_everything(CFG["seed"])
print("device:", DEVICE, "| smoke test:", SMOKE)
print(CFG)

In [ ]:
TEXTS = OUT / "texts.parquet"
IN_WILD = re.compile(r"exploited in the wild|actively exploited|may have been exploited|aware of (?:a )?report",
                     re.IGNORECASE)

def clean(description):
    text = re.sub(r",?\s*as (?:actively )?exploited in the wild[^.]*", "", description or "", flags=re.IGNORECASE)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return " ".join(s for s in sentences if not IN_WILD.search(s)).strip()

if not TEXTS.exists():
    import duckdb
    con = duckdb.connect()
    for t in ["vulnerabilities", "known_exploited", "scoreboard_2026"]:
        con.sql(f'''CREATE VIEW "{t}" AS
                    SELECT * FROM read_csv('{DATA}/{t}.csv', quote='"', escape='"', sample_size=-1)''')
    train = con.sql('''
        WITH v AS (SELECT *, CAST(published AS DATE) AS pub FROM vulnerabilities),
             k AS (SELECT cve_id, MIN(CAST(date_added AS DATE)) AS date_added FROM known_exploited GROUP BY cve_id)
        SELECT v.cve_id, v.assigner, v.description, year(v.pub) AS year, v.pub AS published,
               k.date_added AS kev_date,
               CAST(k.date_added IS NOT NULL AND k.date_added <= DATE '2024-12-31' AS INT) AS y
        FROM v LEFT JOIN k USING (cve_id)
        WHERE v.pub BETWEEN DATE '2018-01-01' AND DATE '2024-12-31'
          AND (k.date_added IS NULL OR k.date_added > last_day(v.pub))
        ORDER BY v.cve_id''').df().assign(split="train")
    val = (pd.read_parquet(DATA / "backtest/val_2025.parquet")
             [["cve_id", "assigner", "description", "y", "epss", "any_exploit", "published", "kev_date_added"]]
             .rename(columns={"kev_date_added": "kev_date"}).assign(split="val", year=2025))
    s26 = con.sql('''SELECT s.cve_id, v.assigner, v.description FROM scoreboard_2026 s
                     LEFT JOIN vulnerabilities v USING (cve_id) ORDER BY s.cve_id''').df().assign(split="score26", year=2026)
    texts = pd.concat([train, val, s26], ignore_index=True)
    texts["published"] = pd.to_datetime(texts["published"])
    texts["kev_date"] = pd.to_datetime(texts["kev_date"])  # kept so later steps can relabel with a later deadline
    texts["text"] = texts["assigner"].fillna("unknown") + " | " + texts["description"].fillna("").map(clean)
    texts.drop(columns=["description"]).to_parquet(TEXTS, index=False)
    print("built", TEXTS.name)

texts = pd.read_parquet(TEXTS)
texts["y"] = texts["y"].fillna(0).astype(int)  # 2026 rows have no label; 0 is a placeholder, never used
train = texts[texts.split == "train"].sort_values("cve_id", ignore_index=True)
val = texts[texts.split == "val"].sort_values("cve_id", ignore_index=True)
score26 = texts[texts.split == "score26"].sort_values("cve_id", ignore_index=True)
print(f"train {len(train):,} rows / {int(train.y.sum())} positives | val {len(val):,} / {int(val.y.sum())} | "
      f"2026 {len(score26):,}")
assert not texts["text"].str.contains(IN_WILD).any()

In [ ]:
BENCH_K, BENCH_N = 500, 72_670
default_k = lambda n: int(round(n * BENCH_K / BENCH_N))

def rank_order(df, score, seed=0):
    cols = [score] if isinstance(score, str) else score
    if isinstance(cols, list) and all(isinstance(c, str) for c in cols):
        keys = [df[c].to_numpy(dtype=float) for c in cols]
    else:
        keys = [np.asarray(score, dtype=float)]
    keys = [np.where(np.isnan(k), -np.inf, k) for k in keys]
    tie = np.random.default_rng(seed).permutation(len(df))
    return np.lexsort([tie] + [-k for k in reversed(keys)])

def hits_at_k(df, score, k=None, seed=0):
    k = k or default_k(len(df))
    return int(df["y"].to_numpy()[rank_order(df, score, seed)[:k]].sum())

def _ranks(df, score, seed=0):
    r = np.empty(len(df), dtype=np.int64)
    r[rank_order(df, score, seed)] = np.arange(len(df))
    return r

def _boot_hits(ranks, y, k, idx):
    return np.array([y[i][np.argpartition(ranks[i], k)[:k]].sum() for i in idx])

def bootstrap(df, score, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    h = _boot_hits(_ranks(df, score, seed), df["y"].to_numpy(), k, idx)
    return int(np.quantile(h, (1 - ci) / 2)), int(np.quantile(h, 1 - (1 - ci) / 2))

def compare(df, score_a, score_b, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    y = df["y"].to_numpy()
    d = (_boot_hits(_ranks(df, score_a, seed), y, k, idx)
         - _boot_hits(_ranks(df, score_b, seed), y, k, idx))
    return {"diff": hits_at_k(df, score_a, k, seed) - hits_at_k(df, score_b, k, seed),
            "range": (int(np.quantile(d, (1 - ci) / 2)), int(np.quantile(d, 1 - (1 - ci) / 2))),
            "p_a_better": round(float((d > 0).mean()), 3)}

BASELINE = ["any_exploit", "epss"]

In [ ]:
tok = AutoTokenizer.from_pretrained(CFG["model"])

def batches(df, batch_size, shuffle=False, seed=0):
    order = np.random.default_rng(seed).permutation(len(df)) if shuffle else np.arange(len(df))
    for i in range(0, len(order), batch_size):
        rows = df.iloc[order[i:i + batch_size]]
        enc = tok(rows["text"].tolist(), truncation=True, max_length=CFG["max_length"], padding=True, return_tensors="pt")
        yield {k: v.to(DEVICE) for k, v in enc.items()}, torch.tensor(rows["y"].fillna(0).to_numpy().astype(np.int64), device=DEVICE)

use_bf16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
autocast = (lambda: torch.autocast("cuda", dtype=torch.bfloat16)) if use_bf16 else (lambda: torch.autocast("cpu", enabled=False))

@torch.no_grad()
def predict(model, df, batch_size=128):
    # P(exploited) for every row. Sorting by length first means less padding, so it's faster.
    model.eval()
    order = df["text"].str.len().sort_values().index.to_numpy()
    out = np.empty(len(df), dtype=np.float32)
    sorted_df = df.loc[order]
    i = 0
    for enc, _ in batches(sorted_df.assign(y=0), batch_size):
        with autocast():
            logits = model(**enc).logits.float()
        p = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        out[i:i + len(p)] = p
        i += len(p)
    result = np.empty(len(df), dtype=np.float32)
    result[np.searchsorted(df.index.to_numpy(), order)] = out
    return result

def train_model(fit, hold, label="model"):
    seed_everything(CFG["seed"])
    model = AutoModelForSequenceClassification.from_pretrained(CFG["model"], num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = CFG["epochs"] * math.ceil(len(fit) / CFG["batch_size"])
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * steps), steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, CFG["pos_weight"]], device=DEVICE))
    log = []
    for epoch in range(CFG["epochs"]):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        for enc, y in batches(fit, CFG["batch_size"], shuffle=True, seed=CFG["seed"] + epoch):
            with autocast():
                loss = loss_fn(model(**enc).logits.float(), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            total += loss.item() * len(y); n += len(y)
        p_hold = predict(model, hold)
        p = np.clip(p_hold.astype(np.float64), 1e-7, 1 - 1e-7)  # weighted log-loss on the held-out rows, on the CPU
        w = np.where(hold.y.to_numpy() == 1, CFG["pos_weight"], 1.0)
        hold_loss = float(-(w * np.where(hold.y.to_numpy() == 1, np.log(p), np.log(1 - p))).sum() / w.sum())
        log.append({"epoch": epoch + 1, "train loss": round(total / n, 4), "held-out loss": round(hold_loss, 4),
                    "held-out AUC": round(float(roc_auc_score(hold.y, p_hold)), 3), "minutes": round((time.time() - t0) / 60, 1)})
        print(f"[{label}]", log[-1])
    return model, pd.DataFrame(log)

## 2. Relabeling for a later deadline

Steps 1-3 label training rows by **December 31, 2024**. The final model may also learn from 2025, with labels by **December 31, 2025**. That adds the 2025 validation rows, and it also turns some older vulnerabilities into positives because CISA added them during 2025.

Every row was already filtered to "not on KEV by the end of its publish month", so a row's label for any deadline is simply "KEV date on or before the deadline."

In [ ]:
def labeled(df, deadline):
    out = df.copy()
    out["y"] = (out["kev_date"].notna() & (out["kev_date"] <= pd.Timestamp(deadline))).astype(int)
    return out

train24 = labeled(train, "2024-12-31")                                   # 2018-2024
train25 = labeled(pd.concat([train, val], ignore_index=True), "2025-12-31").sort_values("cve_id", ignore_index=True)  # 2018-2025
print(f"2018-2024, labels by 2024-12-31: {len(train24):,} rows, {train24.y.sum()} positives")
print(f"2018-2025, labels by 2025-12-31: {len(train25):,} rows, {train25.y.sum()} positives")

## 3. One fine-tuning run = step 3's recipe

`fit_and_score` takes the training rows, keeps all positives plus 20 random negatives each, holds out 10% to watch the loss, fine-tunes, and scores the rows we ask for.

In [ ]:
def fit_and_score(rows, to_score, label, seed=0):
    rng = np.random.default_rng(seed)
    y = rows["y"].to_numpy()
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    if SMOKE:
        pos = rng.choice(pos, min(64, len(pos)), replace=False)
    picked = np.concatenate([pos, rng.choice(neg, CFG["neg_per_pos"] * len(pos), replace=False)])
    ft = rows.iloc[picked].sample(frac=1, random_state=seed).reset_index(drop=True)
    n_hold = int(len(ft) * CFG["holdout"])
    model, log = train_model(ft.iloc[n_hold:].reset_index(drop=True), ft.iloc[:n_hold].reset_index(drop=True), label)
    scores = predict(model, to_score.reset_index(drop=True))
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return scores, log

## 4. The final model and the 2026 scores (saved first)

These files land in order of importance, so a long run is useful even if it's stopped early. First, one model trained on **all** of 2018-2025 (labels by December 31, 2025) scores the 72,670 scoreboard rows. This is the text score that goes into the final ranking.

In [ ]:
HANDOFF = OUT / "handoff"
HANDOFF.mkdir(exist_ok=True)
def save(df, name):
    if SMOKE:
        print(f"(smoke test: {name} not saved)")
    else:
        df.to_csv(HANDOFF / name, index=False)
        print(f"saved {name}: {len(df):,} rows")

t0 = time.time()
existing = HANDOFF / "text_2026.csv"
if existing.exists() and not SMOKE:
    # Built earlier by this same notebook and recipe (that run was paused after saving it); reuse it.
    text_2026, log_c = pd.read_csv(existing), pd.DataFrame()
    print(f"reusing text_2026.csv from an earlier run of this notebook: {len(text_2026):,} rows")
else:
    to_score = score26.iloc[:500] if SMOKE else score26
    s26, log_c = fit_and_score(train25, to_score, "final 2018-2025", seed=CFG["seed"])
    text_2026 = pd.DataFrame({"cve_id": to_score["cve_id"].to_numpy(), "text_score": s26})
    print(f"final model + 2026 scoring: {(time.time() - t0) / 60:.1f} min")
    save(text_2026, "text_2026.csv")
b3 = OUT / "text_val_2025_finetuned.csv"
if b3.exists():
    save(pd.read_csv(b3), "text_val_2025.csv")
text_2026["text_score"].describe()

## 5. Out-of-fold scores for Track A's training set

**Stratified** folds keep the share of positives about equal in each of the 5 parts. Only the rows Track A needs are scored: here, the 2024 rows.

In [ ]:
from sklearn.model_selection import StratifiedKFold

def out_of_fold(rows, score_mask, name, extra=None):
    # extra: optional rows every fold model also scores (here: the 2025 validation set)
    t0 = time.time()
    rows = rows.reset_index(drop=True)
    if SMOKE:  # score only every 50th row in a smoke test
        score_mask = score_mask & (np.arange(len(rows)) % 50 == 0)
    scores = np.full(len(rows), np.nan)
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG["seed"])
    logs, extra_scores = [], []
    for f, (fit_idx, out_idx) in enumerate(folds.split(rows, rows["y"])):
        target = out_idx[score_mask[out_idx]]
        to_score = rows.iloc[target] if extra is None else pd.concat([rows.iloc[target], extra], ignore_index=True)
        s, log = fit_and_score(rows.iloc[fit_idx], to_score, f"{name} fold {f + 1}", seed=CFG["seed"] + f)
        scores[target] = s[:len(target)]
        if extra is not None:
            extra_scores.append(s[len(target):])
        logs.append(log.assign(fold=f + 1))
    print(f"{name}: {(time.time() - t0) / 60:.1f} min")
    out = rows.loc[score_mask, ["cve_id", "y"]].assign(text_score=scores[score_mask])
    assert out["text_score"].notna().all()
    return out, pd.concat(logs), extra_scores

eval_val = val.sample(300, random_state=0).sort_values("cve_id", ignore_index=True) if SMOKE else val
oof_train, log_a, fold_val = out_of_fold(train24, (train24["year"] == 2024).to_numpy(), "oof_train", extra=eval_val)
save(oof_train[["cve_id", "text_score"]], "text_oof_train.csv")

### Sanity check: do the out-of-fold scores rank well?

Out-of-fold scores are honest predictions, so they should rank 2024 positives well, but not suspiciously well. We check on Track A's own 2024 training set (published January-September 2024, labels by December 31, 2024, 33 positives, top 198).

In [ ]:
bt24 = DATA / "backtest/train_2024.parquet"
if bt24.exists():
    ta = pd.read_parquet(bt24).sort_values("cve_id", ignore_index=True)
    ta["text_score"] = ta["cve_id"].map(oof_train.set_index("cve_id")["text_score"])
    print(f"coverage: {ta.text_score.notna().mean():.1%} of Track A's 2024 rows")
    print(f"text OOF hits@{default_k(len(ta))}: {hits_at_k(ta, 'text_score')} | "
          f"baseline: {hits_at_k(ta, BASELINE)} | AUC {roc_auc_score(ta.y, ta.text_score.fillna(0)):.3f}")

## Training luck check: five re-runs of the 2025 test

Step 3 trained **one** model and tested it once on 2025, so its 5 hits@243 could be a lucky or unlucky draw. Each fold model above was trained on 80% of 2018-2024, with nothing from 2025, so each is an independent re-run of that test with different random choices (seed, sampled negatives, training order).

- **The spread across folds** shows how much retraining alone moves the score.
- **The ensemble** averages the five models' scores, which usually steadies the top of the ranking.

In [ ]:
from sklearn.metrics import roc_auc_score
rows = []
for f, s in enumerate(fold_val, 1):
    rows.append({"model": f"fold {f}", f"hits@{default_k(len(eval_val))}": hits_at_k(eval_val, s),
                 "AUC": round(float(roc_auc_score(eval_val.y, s)), 3)})
ens = np.mean([pd.Series(s).rank(pct=True).to_numpy() for s in fold_val], axis=0)  # average of percentile ranks
rows.append({"model": "ensemble of 5", f"hits@{default_k(len(eval_val))}": hits_at_k(eval_val, ens),
             "AUC": round(float(roc_auc_score(eval_val.y, ens)), 3)})
b3 = OUT / "text_val_2025_finetuned.csv"
if b3.exists():
    s3 = pd.read_csv(b3).set_index("cve_id")["text_score"].reindex(eval_val.cve_id).to_numpy()
    rows.append({"model": "step 3 single model", f"hits@{default_k(len(eval_val))}": hits_at_k(eval_val, s3),
                 "AUC": round(float(roc_auc_score(eval_val.y, s3)), 3)})
rows.append({"model": "baseline: exploit code then EPSS", f"hits@{default_k(len(eval_val))}": hits_at_k(eval_val, BASELINE)})
print("ensemble vs. baseline:", compare(eval_val, ens, BASELINE))
if b3.exists():
    print("ensemble vs. step 3 single model:", compare(eval_val, ens, s3))
if not SMOKE:
    pd.DataFrame({"cve_id": eval_val["cve_id"], "text_score": ens}).to_csv(OUT / "text_val_2025_fold_ensemble.csv", index=False)
pd.DataFrame(rows).fillna("")

## 6. Out-of-fold scores for Track A's final training set

The same procedure on 2018-2025 with labels by December 31, 2025, scoring the 2024 and 2025 rows.

In [ ]:
oof_final, log_b, _ = out_of_fold(train25, train25["year"].isin([2024, 2025]).to_numpy(), "oof_final")
save(oof_final[["cve_id", "text_score"]], "text_oof_final.csv")
save(pd.concat([log_c.assign(run="final"), log_a.assign(run="oof_train"), log_b.assign(run="oof_final")]),
     "training_logs.csv")